In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import PROCESSED_CSV, RAW_CSV, VOLATILITY_CSV
from src.extract.data_extractor import DataExtractor
from src.transform.data_transformer import DataTransformer
from src.load.data_loader import DataLoader

extractor = DataExtractor()
transformer = DataTransformer()
loader = DataLoader()

raw_data = extractor.download_data()

# The raw download is indexed by Date, so the index must be written as well.
loader.save_to_csv(raw_data, RAW_CSV, index=True)

data = transformer.clean_data(raw_data)
transformer.validate_data(data)

data = transformer.calculate_daily_returns(data)
data = transformer.normalize_prices(data)
data = transformer.add_asset_class(data)
data = transformer.add_asset_name(data)

loader.save_to_csv(data, PROCESSED_CSV)
loader.save_to_sqlite(data)

In [ ]:
# Volatility = standard deviation of daily returns (second Tableau data source)
volatility_df = transformer.calculate_volatility(data)
loader.save_to_csv(volatility_df, VOLATILITY_CSV)